In [ ]:
import cv2
import numpy as np
import os
import csv
from scipy.ndimage import correlate, generic_laplace

# ============================================
# Métricas de qualidade (mesmas do código anterior)
# ============================================
def psnr(I1, I2):
    '''Peak Signal to Noise Ratio'''
    s1 = cv2.absdiff(I1, I2)
    s1 = np.float32(s1)
    s1 = s1 * s1
    sse = s1.sum()
    if sse <= 1e-10:
        return 0
    else:
        shape = I1.shape
        mse = 1.0 * sse / (shape[0] * shape[1] * shape[2])
        psnr = 10.0 * np.log10((255 * 255) / mse)
        return psnr

def ssim(I1, I2):
    '''Structural Similarity Index'''
    C1 = 6.5025
    C2 = 58.5225
    I1 = np.float32(I1)
    I2 = np.float32(I2)
    I2_2 = I2 * I2
    I1_2 = I1 * I1
    I1_I2 = I1 * I2

    mu1 = cv2.GaussianBlur(I1, (11, 11), 1.5)
    mu2 = cv2.GaussianBlur(I2, (11, 11), 1.5)
    mu1_2 = mu1 * mu1
    mu2_2 = mu2 * mu2
    mu1_mu2 = mu1 * mu2

    sigma1_2 = cv2.GaussianBlur(I1_2, (11, 11), 1.5)
    sigma1_2 -= mu1_2
    sigma2_2 = cv2.GaussianBlur(I2_2, (11, 11), 1.5)
    sigma2_2 -= mu2_2
    sigma12 = cv2.GaussianBlur(I1_I2, (11, 11), 1.5)
    sigma12 -= mu1_mu2

    t1 = 2 * mu1_mu2 + C1
    t2 = 2 * sigma12 + C2
    t3 = t1 * t2
    t1 = mu1_2 + mu2_2 + C1
    t2 = sigma1_2 + sigma2_2 + C2
    t1 = t1 * t2
    ssim_map = cv2.divide(t3, t1)
    return np.mean(ssim_map)

def sam(I1, I2):
    '''Spectral Angle Mapper'''
    def _initial_check(I1, I2):
        if I1.shape != I2.shape:
            raise ValueError("Images must have the same shape")
        return I1, I2

    I1, I2 = _initial_check(I1, I2)
    orig_shape = I1.shape
    I1 = I1.reshape((orig_shape[0] * orig_shape[1], orig_shape[2]))
    I2 = I2.reshape((orig_shape[0] * orig_shape[1], orig_shape[2]))

    sam_angles = np.zeros(I1.shape[1])
    for i in range(I1.shape[1]):
        dot_product = np.dot(I1[:, i], I2[:, i])
        norm_product = np.linalg.norm(I1[:, i]) * np.linalg.norm(I2[:, i])
        val = np.clip(dot_product / (norm_product + 1e-10), -1, 1)
        sam_angles[i] = np.arccos(val)
    return np.mean(sam_angles)

def scc(I1, I2, win=None, ws=8):
    '''Spatial Correlation Coefficient'''
    def _initial_check(I1, I2):
        if I1.shape != I2.shape:
            raise ValueError("Images must have the same shape")
        return I1, I2

    if win is None:
        win = np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]])

    I1, I2 = _initial_check(I1, I2)

    def _scc_single(ch1, ch2):
        # High-pass filter
        hp1 = generic_laplace(ch1.astype(np.float64), lambda in_, ax, out, mode, cval: correlate(in_, win, output=out, mode=mode, cval=cval))
        hp2 = generic_laplace(ch2.astype(np.float64), lambda in_, ax, out, mode, cval: correlate(in_, win, output=out, mode=mode, cval=cval))

        # Uniform filter para variância (simplificado)
        kernel = np.ones((ws, ws)) / (ws * ws)
        mu1 = cv2.filter2D(hp1, -1, kernel)
        mu2 = cv2.filter2D(hp2, -1, kernel)

        sigma1_sq = cv2.filter2D(hp1**2, -1, kernel) - mu1**2
        sigma2_sq = cv2.filter2D(hp2**2, -1, kernel) - mu2**2
        sigma12 = cv2.filter2D(hp1 * hp2, -1, kernel) - mu1 * mu2

        sigma1_sq[sigma1_sq < 0] = 0
        sigma2_sq[sigma2_sq < 0] = 0

        den = np.sqrt(sigma1_sq) * np.sqrt(sigma2_sq) + 1e-10
        scc_map = sigma12 / den
        return np.mean(scc_map)

    coefs = []
    for c in range(I1.shape[2]):
        coefs.append(_scc_single(I1[:, :, c], I2[:, :, c]))
    return np.mean(coefs)

# ============================================
# Parâmetros e listas de nomes
# ============================================
original_image_path = 'foto_grupo_2.png'  # imagem original limpa
src = cv2.imread(original_image_path)
if src is None:
    raise Exception(f"Imagem original não encontrada: {original_image_path}")

# Bases (imagens de entrada para os filtros)
bases = ['original', 'gauss', 'sp']
# Filtros aplicados
filtros = ['media', 'gaussian', 'median', 'bilateral']
# Tamanhos de kernel
kernels = [3, 5, 11, 29]

# Arquivo de saída CSV
csv_filename = 'metricas_imagens_filtradas.csv'

# ============================================
# Processar cada combinação e calcular métricas
# ============================================
with open(csv_filename, 'w', newline='') as csvfile:
    fieldnames = ['base_image', 'filter', 'kernel', 'psnr', 'ssim', 'sam', 'scc']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    # Primeiro, calcular métricas para as imagens ruidosas (sem filtro)
    noisy_images = [
        ('gauss', 'ruido_gaussiano.jpg'),
        ('sp', 'ruido_salt_pepper.jpg')
    ]
    for base_name, filename in noisy_images:
        img = cv2.imread(filename)
        if img is not None:
            p = psnr(src, img)
            s = ssim(src, img)
            sa = sam(src, img)
            sc = scc(src, img)
            writer.writerow({
                'base_image': base_name,
                'filter': 'none',
                'kernel': 0,
                'psnr': f"{p:.2f}",
                'ssim': f"{s:.4f}",
                'sam': f"{sa:.4f}",
                'scc': f"{sc:.4f}"
            })
            print(f"Processado: {filename}")
        else:
            print(f"Aviso: {filename} não encontrado")

    # Agora processar todas as imagens filtradas
    for base in bases:
        for filtro in filtros:
            for k in kernels:
                filename = f"{filtro}_{base}_{k}x{k}.jpg"
                img = cv2.imread(filename)
                if img is not None:
                    p = psnr(src, img)
                    s = ssim(src, img)
                    sa = sam(src, img)
                    sc = scc(src, img)
                    writer.writerow({
                        'base_image': base,
                        'filter': filtro,
                        'kernel': k,
                        'psnr': f"{p:.2f}",
                        'ssim': f"{s:.4f}",
                        'sam': f"{sa:.4f}",
                        'scc': f"{sc:.4f}"
                    })
                    print(f"Processado: {filename}")
                else:
                    print(f"Aviso: {filename} não encontrado")

print(f"\nProcessamento concluído! Resultados salvos em '{csv_filename}'")